In [24]:
# ==============================
# BLOCK 1 — Imports + Paths
# ==============================
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import flopy
from flopy.mf6.utils.postprocessing import get_structured_faceflows

BASE   = Path(r"C:\Jupyterbook\Working\Geostat_Real\Scenario_A_nomask_30")
PAIRED = BASE / "paired"

OUT_DIR  = BASE / "Results" / "metrics_tables"
FIGS_DIR = BASE / "Figures" / "Results_Summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

pairs = sorted([d for d in PAIRED.iterdir() if d.is_dir() and re.match(r"R\d{3}$", d.name, re.IGNORECASE)])
print("Pairs found:", len(pairs))

Pairs found: 30


In [25]:
# ==============================
# BLOCK 2 — Grid + Settings
# ==============================
nlay = 30
nrow = 1
ncol = 240
system_length = 24_000.0
delr = system_length / ncol   # 100 m

x_km = ((np.arange(ncol) + 0.5) * delr) / 1000.0

THRESH  = 25.0
T_FINAL = 2_920_000.0  # <-- your final totim

TRANSECT_X_KM = 16.0

def right_face_col_for_x(x_km: float, delr_m: float, ncol: int) -> int:
    """
    Return the 0-based column index j whose RIGHT FACE lies at x_km.
    For uniform delr, right face of column j is at x=(j+1)*delr.
    """
    x_m = float(x_km) * 1000.0
    j = int(np.floor(x_m / delr_m) - 1)
    return max(0, min(ncol - 2, j))

transect_face_col = right_face_col_for_x(TRANSECT_X_KM, delr, ncol)
transect_face_x_km = ((transect_face_col + 1) * delr) / 1000.0

print("delr (m):", delr)
print("transect_face_col:", transect_face_col, "| right-face x_km:", transect_face_x_km)
print("THRESH:", THRESH, "| T_FINAL:", T_FINAL)


delr (m): 100.0
transect_col: 160 | x_km: 16.05
THRESH: 25.0 | T_FINAL: 2920000.0


In [26]:
# ==============================
# BLOCK 4 — Toe (your working method)
# ==============================
def brine_toe_from_trans_ucn(trans_ucn: Path, t_final: float, thresh: float) -> int | None:
    hf = flopy.utils.HeadFile(str(trans_ucn), text="CONCENTRATION")
    c = np.asarray(hf.get_data(totim=t_final)).squeeze()

    if c.ndim == 1:
        cols = c >= thresh
    elif c.ndim == 2:
        cols = np.any(c >= thresh, axis=0)
    elif c.ndim == 3:
        cols = np.any(c >= thresh, axis=(0, 1))
    else:
        raise ValueError(f"Unexpected concentration shape: {c.shape}")

    if not np.any(cols):
        return None

    return int(np.where(cols)[0].max()) + 1  # 1-based col index

def toe_km_from_run(run_dir: Path, t_final: float, thresh: float, delr_m: float) -> float:
    toe_col_1based = brine_toe_from_trans_ucn(run_dir / "trans.ucn", t_final, thresh)
    if toe_col_1based is None:
        return float("nan")
    return float(((toe_col_1based - 1) * delr_m) / 1000.0)

In [27]:
# ==============================
# BLOCK 5 — Qin from FLOW-JA-FACE via structured faceflows (FRF)
# ==============================
def require(run_dir: Path, fname: str) -> Path:
    p = run_dir / fname
    if not p.exists():
        raise FileNotFoundError(f"Missing {fname} in {run_dir}")
    return p

def find_bud(run_dir: Path) -> Path:
    cands = [run_dir / "flow.bud", run_dir / "flow.cbb"]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError(f"No budget file found in {run_dir} (looked for flow.bud/flow.cbb)")

def find_grb(run_dir: Path) -> Path:
    cands = sorted(run_dir.glob("*.dis.grb")) + sorted(run_dir.glob("*.grb"))
    if cands:
        return cands[0]
    raise FileNotFoundError(f"No GRB file found in {run_dir}")

def open_budget(bud_path: Path):
    try:
        return flopy.utils.CellBudgetFile(str(bud_path), precision="double")
    except Exception:
        return flopy.utils.CellBudgetFile(str(bud_path), precision="single")

def choose_closest_time(target_t: float, available_times) -> float:
    times = np.array(available_times, dtype=float)
    if times.size == 0:
        raise RuntimeError("No times found in budget file.")
    return float(times[np.argmin(np.abs(times - float(target_t)))])

def qin_from_flowja_frf(run_dir: Path, transect_face_col: int, nlay: int, nrow: int, ncol: int) -> float:
    """
    Convert FLOW-JA-FACE to structured FRF and compute Qin at the transect face.

    FRF[k, i, j] is flow across the RIGHT face of cell (k, i, j).
    Qin at x=16 km is computed as sum(max(FRF[k, 0, j], 0)) over layers,
    where j is the column whose right face is located at x=16 km.
    """
    hds = require(run_dir, "flow.hds")
    bud = find_bud(run_dir)
    grb = find_grb(run_dir)

    hf = flopy.utils.HeadFile(str(hds))
    t_h = float(hf.get_times()[-1])

    cbb = open_budget(bud)
    t_b = choose_closest_time(t_h, cbb.get_times())

    flowja_list = cbb.get_data(text="FLOW-JA-FACE", totim=t_b)
    if not flowja_list:
        raise RuntimeError(f"No FLOW-JA-FACE in {bud.name} at totim={t_b}")
    flowja = flowja_list[0]

    frf, fff, flf = get_structured_faceflows(flowja, grb_file=str(grb))

    if frf.shape != (nlay, nrow, ncol):
        raise RuntimeError(f"Unexpected FRF shape {frf.shape}, expected {(nlay,nrow,ncol)}")

    q_layers = frf[:, 0, transect_face_col]
    return float(np.sum(np.clip(q_layers, 0.0, None)))


In [28]:
# ==============================
# BLOCK 6 — Single-run sanity test
# ==============================
test_pair = pairs[0]  # or pick: PAIRED/"R015"
U = test_pair / "unmasked"

toe_test = toe_km_from_run(U, T_FINAL, THRESH, delr)
qin_test = qin_from_flowja_frf(U, transect_face_col, nlay=nlay, nrow=nrow, ncol=ncol)

print("Test pair:", test_pair.name)
print("Toe km (unmasked):", toe_test)
print(f"Qin at x={transect_face_x_km:.2f} km (unmasked):", qin_test)
print("GRB used:", find_grb(U).name)
print("BUD used:", find_bud(U).name)

NameError: name 'find_grb' is not defined